# Clean spike experiments

This notebook keeps the two experiments separate:

1. 100-class mean-waveform overfit test
2. Binary spike-vs-background detection test

Do not mix these tasks in one training run. The earlier notebook failed because it switched between incompatible class definitions and reused checkpoints across different tasks.

In [ ]:
from pathlib import Path
import random

ROOT = Path('/Users/Zita/Documents/EGYETEM/Önlab/yolo26')
MEAN_WAVEFORM_DATASET = ROOT / 'DATASETS' / 'yolo26_mean_waveform_overfit'
BINARY_DATASET = ROOT / 'DATASETS' / 'yolo26_spike_detection_100'

print('ROOT:', ROOT)
print('100-class dataset:', MEAN_WAVEFORM_DATASET)
print('Binary dataset:', BINARY_DATASET)

In [ ]:
from pathlib import Path

def check_dataset_consistency(dataset_dir: Path):
    data_yaml = dataset_dir / 'data.yaml'
    assert data_yaml.exists(), f'Missing data.yaml in {dataset_dir}'
    
    train_images = sorted((dataset_dir / 'images' / 'train').glob('*.png'))
    val_images = sorted((dataset_dir / 'images' / 'val').glob('*.png'))
    train_labels = sorted((dataset_dir / 'labels' / 'train').glob('*.txt'))
    val_labels = sorted((dataset_dir / 'labels' / 'val').glob('*.txt'))
    
    print(f'
Dataset: {dataset_dir}')
    print(f'train images: {len(train_images)}')
    print(f'val images:   {len(val_images)}')
    print(f'train labels: {len(train_labels)}')
    print(f'val labels:   {len(val_labels)}')
    
    train_stems = {p.stem for p in train_images}
    train_label_stems = {p.stem for p in train_labels}
    val_stems = {p.stem for p in val_images}
    val_label_stems = {p.stem for p in val_labels}
    
    print('Train image/label match:', train_stems == train_label_stems)
    print('Val image/label match:  ', val_stems == val_label_stems)
    print('Train/val overlap:      ', bool(train_stems & val_stems))
    
    print(data_yaml.read_text())

check_dataset_consistency(MEAN_WAVEFORM_DATASET)
check_dataset_consistency(BINARY_DATASET)

## Experiment 1: 100-class mean-waveform overfit test

This is a strict overfit test. Keep it separate from binary spike detection. The goal is to see whether the model can memorize the 100 waveform classes, not whether it generalizes to a binary spike/no-spike problem.

In [ ]:
from ultralytics import YOLO

model_100 = YOLO('yolo26n.yaml')

results_100 = model_100.train(
    data=str(MEAN_WAVEFORM_DATASET / 'data.yaml'),
    imgsz=128,
    epochs=200,
    batch=16,
    seed=42,
    device='mps',
    workers=0,
    cache=True,
    patience=50,
    project='runs/spike_yolo_overfit',
    name='mean_waveforms_100_overfit_clean',
    val=True,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    close_mosaic=0,
)

print('100-class experiment finished.')

## Experiment 2: binary spike detection

This should be a different dataset and a different task. Use a single class called `spike`, and keep the positive/negative examples separate from the 100-class waveform identity experiment.

In [ ]:
from pathlib import Path
from PIL import Image
import shutil

def build_binary_spike_dataset(source_dataset: Path, output_dir: Path, train_ratio: float = 0.8, seed: int = 42):
    random.seed(seed)
    output_dir.mkdir(parents=True, exist_ok=True)

    for split in ['train', 'val']:
        (output_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    source_images = sorted((source_dataset / 'images' / 'train').glob('*.png'))
    source_labels = sorted((source_dataset / 'labels' / 'train').glob('*.txt'))

    if len(source_images) != len(source_labels):
        raise ValueError('Image/label count mismatch in source dataset.')

    items = []
    for img_path, label_path in zip(source_images, source_labels):
        items.append({'type': 'positive', 'image': img_path, 'label': label_path})
    
    for _ in range(len(items)):
        items.append({'type': 'negative', 'image': None, 'label': None})

    random.shuffle(items)
    split_idx = int(len(items) * train_ratio)
    train_items = items[:split_idx]
    val_items = items[split_idx:]

    def create_blank_image(path: Path, gray_value: int = 128):
        img = Image.new('L', (128, 128), color=gray_value)
        img.save(path)

    def process_split(split_name: str, chunk):
        for index, item in enumerate(chunk):
            output_name = f'spike_{index:04d}'
            if item['type'] == 'positive':
                out_img = output_dir / 'images' / split_name / f'{output_name}.png'
                out_label = output_dir / 'labels' / split_name / f'{output_name}.txt'
                shutil.copy2(item['image'], out_img)

                with open(item['label']) as f:
                    lines = []
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cls, x, y, w, h = parts
                            lines.append(f'0 {x} {y} {w} {h}\n')
                if not lines:
                    raise ValueError(f'Empty label: {item[label]}')
                with open(out_label, 'w') as f:
                    f.writelines(lines)
            else:
                out_img = output_dir / 'images' / split_name / f'{output_name}.png'
                out_label = output_dir / 'labels' / split_name / f'{output_name}.txt'
                create_blank_image(out_img, gray_value=128)
                out_label.touch()

    process_split('train', train_items)
    process_split('val', val_items)

    yaml_path = output_dir / 'data.yaml'
    yaml_path.write_text(f'''path: {output_dir}\ntrain: images/train\nval: images/val\n\nnames:\n  0: spike\n''')
    return output_dir

BINARY_DATASET_CLEAN = ROOT / 'DATASETS' / 'yolo26_spike_detection_binary_clean'
BINARY_DATASET_CLEAN = build_binary_spike_dataset(MEAN_WAVEFORM_DATASET, BINARY_DATASET_CLEAN)
print('Binary dataset created:', BINARY_DATASET_CLEAN)
print((BINARY_DATASET_CLEAN / 'data.yaml').read_text())

In [ ]:
from ultralytics import YOLO

model_binary = YOLO('yolo26n.yaml')

results_binary = model_binary.train(
    data=str(BINARY_DATASET_CLEAN / 'data.yaml'),
    imgsz=128,
    epochs=200,
    batch=16,
    seed=42,
    device='mps',
    workers=0,
    cache=True,
    patience=50,
    project='runs/spike_yolo_overfit',
    name='spike_binary_clean',
    val=True,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    close_mosaic=0,
)

print('Binary spike experiment finished.')

## Interpretation

- The 100-class run tests identity recognition over 100 waveform classes.
- The binary spike run tests spike detection with a single class.
- Do not compare these directly unless they are run on the same task definition.
- The bad early results were caused by task mixing, checkpoint reuse, and dataset inconsistencies, not by a single bad model config alone.